# 09 Stage 2 Logistic Regression — Health Outcome Prediction

**Owner:** PBC  
**Targets:** `target_unmet_fp` (and `target_anc_gap` when m14 is available)  
**Depends on:** `07_data_integration.ipynb`, `08_clustering.ipynb`

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.stage2_logistic import configure_logging, train_all_stage2_logistic
from src.models.stage2_xgboost import STAGE2_TARGETS, TARGET_DISPLAY, load_stage2_data

configure_logging()

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
# Ensure Stage 2 inputs exist
stage2_files = [
    PROJECT_ROOT / 'data/processed/stage2/X_stage2_preclustering.csv',
    PROJECT_ROOT / 'data/processed/stage2/y_stage2_targets.csv',
    PROJECT_ROOT / 'outputs/stage2_results/cluster_assignments.csv',
]
if not all(p.exists() for p in stage2_files):
    raise FileNotFoundError('Run scripts/run_stage2_data_prep.py or 08_clustering.ipynb first.')
print('Stage 2 inputs found.')

Stage 2 inputs found.


In [3]:
X_full, y = load_stage2_data()
print(f'Feature matrix (with cluster dummies): {X_full.shape}')
for col in y.columns:
    nn = y[col].notna().sum()
    pos = y.loc[y[col].notna(), col].sum() if nn else 0
    print(f'{col}: non-null N = {nn:,} | positive = {int(pos):,}')

2026-07-24 18:55:11,700 | INFO | src.models.stage2_xgboost | Loaded Stage 2 data: X=(724115, 43), targets=['target_unmet_fp', 'target_anc_gap']


Feature matrix (with cluster dummies): (724115, 43)
target_unmet_fp: non-null N = 466,859 | positive = 49,672
target_anc_gap: non-null N = 163,018 | positive = 61,585


In [4]:
# Train separate LogisticRegression models per target (Section 6.2)
results = train_all_stage2_logistic()
metrics_df = results.get('metrics_df')
if metrics_df is not None:
    display_cols = ['Target', 'TrainSize', 'TestSize', 'ROC-AUC', 'F1-Score', 'CV_ROC-AUC', 'Barrier_Uplift']
    metrics_df[[c for c in display_cols if c in metrics_df.columns]]

2026-07-24 18:55:18,134 | INFO | src.models.stage2_xgboost | Loaded Stage 2 data: X=(724115, 43), targets=['target_unmet_fp', 'target_anc_gap']
2026-07-24 18:55:18,295 | INFO | src.models.stage2_xgboost | target_anc_gap — analytic sample: 163018 rows (positive rate 0.3778)


--- Top predictors for target_anc_gap ---
                             Feature  Coefficient  OddsRatio
              household_barrier_prob     0.268659   1.308208
                 vulnerability_score     0.113944   1.120689
             composite_barrier_score     0.076440   1.079437
                                v013     0.058655   1.060409
                       v743f_missing     0.052684   1.054096
                      v130_christian     0.043250   1.044199
v743f_respondent and husband/partner     0.036699   1.037381
                    v717_not working     0.036400   1.037071
                        v717_missing     0.022217   1.022466
                          v130_other     0.018641   1.018816

=== Logistic Regression | target_anc_gap barrier ===
  Model       : Logistic Regression
  Target      : target_anc_gap
  Accuracy    : 0.5976
  ROC-AUC     : 0.6356
  Precision   : 0.4746
  Recall      : 0.608
  F1-Score    : 0.5331
              precision    recall  f1-score   suppor

2026-07-24 18:56:33,344 | INFO | src.models.stage2_xgboost | target_unmet_fp — analytic sample: 466859 rows (positive rate 0.1064)


--- Top predictors for target_unmet_fp ---
               Feature  Coefficient  OddsRatio
          v501_married     0.322195   1.380153
         v743f_missing     0.213869   1.238460
household_barrier_prob     0.142312   1.152936
                  v106     0.135046   1.144589
      v717_not working     0.100663   1.105904
        v130_christian     0.079399   1.082636
                  v013     0.072379   1.075062
   vulnerability_score     0.057769   1.059470
            v131_tribe     0.043948   1.044928
           v130_muslim     0.039469   1.040259

=== Logistic Regression | target_unmet_fp barrier ===
  Model       : Logistic Regression
  Target      : target_unmet_fp
  Accuracy    : 0.5841
  ROC-AUC     : 0.6591
  Precision   : 0.157
  Recall      : 0.6658
  F1-Score    : 0.2541
              precision    recall  f1-score   support

           0       0.94      0.57      0.71     83438
           1       0.16      0.67      0.25      9934

    accuracy                           

2026-07-24 18:59:37,038 | INFO | src.models.stage2_logistic | Saved evaluation metrics -> B:\PBCS\Major Project\BarrierLens_MP_G25_P48\outputs\stage2_results\logistic_evaluation_results.csv


target_unmet_fp: socioeconomic-only=0.6567 | +barriers=0.6583 | uplift=+0.0017


In [5]:
# Top odds-ratio predictors per target
for target_col in STAGE2_TARGETS:
    if target_col not in results.get('targets', {}):
        print(f'Skipped {TARGET_DISPLAY.get(target_col, target_col)} (no analytic sample)')
        continue
    coefs = results['targets'][target_col]['coefficients']
    print(f'\n=== {TARGET_DISPLAY.get(target_col, target_col)} — top odds ratios ===')
    print(coefs.head(10).to_string(index=False))


=== ANC Care Gap — top odds ratios ===
                             Feature  Coefficient  OddsRatio
              household_barrier_prob     0.268659   1.308208
                 vulnerability_score     0.113944   1.120689
             composite_barrier_score     0.076440   1.079437
                                v013     0.058655   1.060409
                       v743f_missing     0.052684   1.054096
                      v130_christian     0.043250   1.044199
v743f_respondent and husband/partner     0.036699   1.037381
                    v717_not working     0.036400   1.037071
                        v717_missing     0.022217   1.022466
                          v130_other     0.018641   1.018816

=== Unmet Family Planning Need — top odds ratios ===
               Feature  Coefficient  OddsRatio
          v501_married     0.322195   1.380153
         v743f_missing     0.213869   1.238460
household_barrier_prob     0.142312   1.152936
                  v106     0.135046   1.144589
